# Lab: Same prompt, different answers
**Solution notebook.** Reference solution. Try the starter notebook first.

## What you will do
You will compare answers to the same prompt, run three times in two different models, and measure how much they change. You will also check whether each answer kept the facts it was given.

**Time:** 20 to 25 minutes. **Needs:** Google Colab or Python 3.12+. No API key and no extra libraries.

**Before the lab (5 minutes, optional):** open Google AI Studio (https://aistudio.google.com) and one other AI chat tool you are allowed to use at work. Paste the prompt below into each tool three times, starting a new chat each time, and copy each answer into `MY_RESPONSES` in Step 1. If you skip this, the notebook runs on sample answers.

> Write a one-sentence subject line for an email telling customers that online banking will be down for 2 hours on Saturday from 1 a.m. to 3 a.m. for maintenance.

In [ ]:
# Step 1. The answers to compare.
# Sample answers written for this course. Replace them with your own runs if you have them.
SAMPLE_RESPONSES = [
    ("Model A", 1, "Scheduled maintenance: online banking unavailable Saturday 1 a.m. to 3 a.m."),
    ("Model A", 2, "Heads up: online banking will be down this Saturday from 1 to 3 a.m."),
    ("Model A", 3, "Planned maintenance this Saturday, 1 a.m. to 3 a.m.: online banking offline for 2 hours"),
    ("Model B", 1, "Important: Online Banking Maintenance This Weekend"),
    ("Model B", 2, "Online banking will be briefly unavailable this weekend for upgrades"),
    ("Model B", 3, "Saturday 1-3 a.m.: online banking maintenance"),
]

MY_RESPONSES = [
    # ("Model A", 1, "paste answer here"),
]

responses = MY_RESPONSES if MY_RESPONSES else SAMPLE_RESPONSES
print(f"Comparing {len(responses)} answers from {len({r[0] for r in responses})} models")

In [ ]:
# Step 2. Measure how similar two answers are, from 0.0 (nothing shared) to 1.0 (identical).
from difflib import SequenceMatcher
from itertools import combinations

def similarity(a: str, b: str) -> float:
    """Return the similarity of two answers, ignoring capital letters, rounded to 2 decimals."""
    return round(SequenceMatcher(None, a.lower(), b.lower()).ratio(), 2)

print(similarity("Saturday maintenance", "saturday maintenance"))  # expect 1.0

In [ ]:
# Step 3. Did each answer keep the facts in the prompt?
REQUIRED_FACTS = {
    "day": ["saturday"],
    "time": ["1 a.m.", "1 to 3", "1-3"],
}

def facts_kept(answer: str) -> dict:
    """Return {fact_name: True/False}. A fact is kept if any of its phrases appears in the answer."""
    text = answer.lower()
    return {name: any(p in text for p in phrases) for name, phrases in REQUIRED_FACTS.items()}

print(facts_kept("Saturday 1-3 a.m.: online banking maintenance"))

In [ ]:
# Step 4. Report: consistency within each model, and facts kept in each answer.
for model in sorted({m for m, _, _ in responses}):
    texts = [t for m, _, t in responses if m == model]
    pairs = [similarity(a, b) for a, b in combinations(texts, 2)]
    avg = round(sum(pairs) / len(pairs), 2) if pairs else 1.0
    print(f"\n{model}: average similarity between its own runs = {avg}")
    for m, run, t in responses:
        if m == model:
            missing = [k for k, v in facts_kept(t).items() if not v]
            status = "all facts kept" if not missing else "MISSING: " + ", ".join(missing)
            print(f"  run {run}: {len(t.split()):>2} words | {status}")

between = [similarity(a[2], b[2]) for a, b in combinations(responses, 2) if a[0] != b[0]]
print(f"\nAverage similarity between models = {round(sum(between) / len(between), 2)}")

## Check your understanding
1. Why did the same model give different answers to the same prompt?
2. Which answers dropped a fact? Would you have noticed without the check?

## Apply it to your field
Write one prompt from your own work (your domain track). List two facts every answer must keep. Run it three times and add those facts to `REQUIRED_FACTS`. In your project log, note one risk if a dropped fact reached a customer, patient, student or client.

---
BJS Academy · Prompt to Agent (Course 03) · Dr. Bih Janet Shufor. Original course material. Learn AI. Shape tomorrow.